In [18]:
import asyncio

async def stuck_model():
    await asyncio.sleep(10)
    return "too late"

async def main():
    try:
        answer = await asyncio.wait_for(stuck_model(), timeout=0.5)
    except asyncio.TimeoutError:
        print("no answer after 0.5 seconds, moving on")

In [19]:
await main()

no answer after 0.5 seconds, moving on


In [20]:
class FlakyModel:
    def __init__(self, failures):
        self.failures = failures
        self.calls = 0
    async def ask(self):
        self.calls += 1
        await asyncio.sleep(0.1)
        if self.calls <= self.failures:
            raise ConnectionError("the server did not answer")
        return {"category":"billing", "priority":4}

In [12]:
async def main():
    model = FlakyModel(failures=2)
    for _ in range(3):
        try:
            print(await model.ask())
        except ConnectionError as e:
            print(e)

await main()

the server did not answer
the server did not answer
{'category': 'billing', 'priority': 4}


In [26]:
async def ask_with_retries(model, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            print(f"Attempt no is {attempt}")
            return await model.ask()
        except ConnectionError as e:
            print(f"attempt {attempt} failed: {e}")
            await asyncio.sleep(0.2 * attempt)
    raise ConnectionError(f"gave up after {attempts} retries")

In [27]:
async def main():
    print(await ask_with_retries(FlakyModel(failures=3), 4))

await main()

Attempt no is 1
attempt 1 failed: the server did not answer
Attempt no is 2
attempt 2 failed: the server did not answer
Attempt no is 3
attempt 3 failed: the server did not answer
Attempt no is 4
{'category': 'billing', 'priority': 4}


In [29]:
async def main():
    print(await ask_with_retries(FlakyModel(failures=5),4))

await main()

Attempt no is 1
attempt 1 failed: the server did not answer
Attempt no is 2
attempt 2 failed: the server did not answer
Attempt no is 3
attempt 3 failed: the server did not answer
Attempt no is 4
attempt 4 failed: the server did not answer


ConnectionError: gave up after 4 retries

In [33]:
async def main():
    print(await ask_with_retries(FlakyModel(failures=3),3))

await main()

Attempt no is 1
attempt 1 failed: the server did not answer
Attempt no is 2
attempt 2 failed: the server did not answer
Attempt no is 3
attempt 3 failed: the server did not answer


ConnectionError: gave up after 3 retries

In [34]:
async def main():
    print(await ask_with_retries(FlakyModel(failures=3),4))

await main()

Attempt no is 1
attempt 1 failed: the server did not answer
Attempt no is 2
attempt 2 failed: the server did not answer
Attempt no is 3
attempt 3 failed: the server did not answer
Attempt no is 4
{'category': 'billing', 'priority': 4}


In [38]:
async def main():
    print(await ask_with_retries(FlakyModel(failures=5),6))

await main()

Attempt no is 1
attempt 1 failed: the server did not answer
Attempt no is 2
attempt 2 failed: the server did not answer
Attempt no is 3
attempt 3 failed: the server did not answer
Attempt no is 4
attempt 4 failed: the server did not answer
Attempt no is 5
attempt 5 failed: the server did not answer
Attempt no is 6
{'category': 'billing', 'priority': 4}


In [39]:
async def ask_with_retries(model, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return await asyncio.wait_for(model.ask(), timeout=0.05)
        except TimeoutError as error:
            print(f"attempt {attempt} failed: {error!r}")
            await asyncio.sleep(0.2 * attempt)

In [41]:
async def main():
    print(await ask_with_retries(FlakyModel(failures=4),6))

await main()

attempt 1 failed: TimeoutError()
attempt 2 failed: TimeoutError()
attempt 3 failed: TimeoutError()
attempt 4 failed: TimeoutError()
attempt 5 failed: TimeoutError()
attempt 6 failed: TimeoutError()
None


In [43]:
async def main():
    print(await ask_with_retries(FlakyModel(failures=0),4))

await main()

attempt 1 failed: TimeoutError()
attempt 2 failed: TimeoutError()
attempt 3 failed: TimeoutError()
attempt 4 failed: TimeoutError()
None
